In [ ]:
# ============================================================
# ISOLATION FOREST ANOMALY DETECTION
# USING EXISTING TRAIN / VAL / TEST SPLITS
# ============================================================

import pandas as pd
import numpy as np

from sklearn.ensemble import IsolationForest

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

print("\n" + "=" * 70)
print("ISOLATION FOREST ANOMALY DETECTION")
print("=" * 70)


# ============================================================
# 1. LOAD DATASET
# ============================================================

df = pd.read_csv("ml_features_and_labels.csv")

print("\nDataset loaded successfully.")
print("Dataset shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())


# ============================================================
# 2. DEFINE IMPORTANT COLUMNS
# ============================================================

TARGET_COLUMN = "label"
SPLIT_COLUMN = "split"

# These columns are identifiers/metadata and should not
# be used as Isolation Forest features.
DROP_COLUMNS = [
    "label",
    "split",
    "taxonomy",
    "ID"
]


# ============================================================
# 3. CHECK REQUIRED COLUMNS
# ============================================================

required_columns = [
    TARGET_COLUMN,
    SPLIT_COLUMN
]

for column in required_columns:
    if column not in df.columns:
        raise ValueError(
            f"Required column '{column}' was not found."
        )


# ============================================================
# 4. CHECK LABEL DISTRIBUTION
# ============================================================

print("\n" + "=" * 70)
print("LABEL DISTRIBUTION")
print("=" * 70)

print(
    df[TARGET_COLUMN]
    .value_counts()
    .sort_index()
)


# ============================================================
# 5. CHECK EXISTING SPLITS
# ============================================================

print("\n" + "=" * 70)
print("DATASET SPLITS")
print("=" * 70)

print(
    df[SPLIT_COLUMN]
    .value_counts()
)


print("\nLabel distribution by split:")

print(
    pd.crosstab(
        df[SPLIT_COLUMN],
        df[TARGET_COLUMN]
    )
)


# ============================================================
# 6. SEPARATE TRAIN / VALIDATION / TEST
# ============================================================

train_df = df[
    df[SPLIT_COLUMN].str.lower() == "train"
].copy()

val_df = df[
    df[SPLIT_COLUMN].str.lower() == "val"
].copy()

test_df = df[
    df[SPLIT_COLUMN].str.lower() == "test"
].copy()


print("\n" + "=" * 70)
print("SPLIT SIZES")
print("=" * 70)

print("Training   :", train_df.shape)
print("Validation :", val_df.shape)
print("Test       :", test_df.shape)


# ============================================================
# 7. CREATE FEATURES AND TARGET
# ============================================================

feature_columns = [
    column
    for column in df.columns
    if column not in DROP_COLUMNS
]


print("\nNumber of feature columns:", len(feature_columns))

print("\nFeature columns:")
for column in feature_columns:
    print(" -", column)


X_train = train_df[feature_columns].copy()
y_train = train_df[TARGET_COLUMN].copy()

X_val = val_df[feature_columns].copy()
y_val = val_df[TARGET_COLUMN].copy()

X_test = test_df[feature_columns].copy()
y_test = test_df[TARGET_COLUMN].copy()


# ============================================================
# 8. CONVERT FEATURES TO NUMERIC
# ============================================================

# Your dataset contains boolean features such as:
#
# True / False
#
# Convert them to:
#
# 1 / 0

for X in [X_train, X_val, X_test]:

    bool_columns = X.select_dtypes(
        include=["bool"]
    ).columns

    X[bool_columns] = X[bool_columns].astype(int)


# Convert everything to numeric
X_train = X_train.apply(
    pd.to_numeric,
    errors="coerce"
)

X_val = X_val.apply(
    pd.to_numeric,
    errors="coerce"
)

X_test = X_test.apply(
    pd.to_numeric,
    errors="coerce"
)


# ============================================================
# 9. HANDLE MISSING VALUES
# ============================================================

# Use training medians only.
#
# This avoids using information from validation/test data
# during preprocessing.

train_medians = X_train.median()

X_train = X_train.fillna(train_medians)
X_val = X_val.fillna(train_medians)
X_test = X_test.fillna(train_medians)


# ============================================================
# 10. CONVERT LABELS TO INTEGER
# ============================================================

y_train = pd.to_numeric(
    y_train,
    errors="coerce"
).astype(int)

y_val = pd.to_numeric(
    y_val,
    errors="coerce"
).astype(int)

y_test = pd.to_numeric(
    y_test,
    errors="coerce"
).astype(int)


# ============================================================
# 11. DISPLAY FINAL DATA SHAPES
# ============================================================

print("\n" + "=" * 70)
print("FINAL DATA SHAPES")
print("=" * 70)

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_val  :", X_val.shape)
print("y_val  :", y_val.shape)

print("X_test :", X_test.shape)
print("y_test :", y_test.shape)


# ============================================================
# 12. LABEL DISTRIBUTION
# ============================================================

print("\nTraining labels:")
print(
    y_train.value_counts()
    .sort_index()
    .rename(
        index={
            0: "Normal",
            1: "Anomaly"
        }
    )
)

print("\nValidation labels:")
print(
    y_val.value_counts()
    .sort_index()
    .rename(
        index={
            0: "Normal",
            1: "Anomaly"
        }
    )
)

print("\nTest labels:")
print(
    y_test.value_counts()
    .sort_index()
    .rename(
        index={
            0: "Normal",
            1: "Anomaly"
        }
    )
)


# ============================================================
# 13. BASELINE ISOLATION FOREST
# ============================================================
#
# Isolation Forest is UNSUPERVISED.
#
# Therefore, y_train is NOT given to the model.
#
# The model learns only from X_train.
#
# Isolation Forest prediction:
#
#       1  = normal
#      -1  = anomaly
#
# We convert this to:
#
#       0  = normal
#       1  = anomaly
# ============================================================

iso_forest = IsolationForest(
    n_estimators=200,
    contamination="auto",
    max_samples="auto",
    max_features=1.0,
    random_state=42,
    n_jobs=-1
)


print("\n" + "=" * 70)
print("MODEL PARAMETERS")
print("=" * 70)

print("n_estimators  :", 200)
print("contamination :", "auto")
print("max_samples   :", "auto")
print("max_features  :", 1.0)
print("random_state  :", 42)


# ============================================================
# 14. TRAIN MODEL
# ============================================================

iso_forest.fit(X_train)

print("\nIsolation Forest training completed.")


# ============================================================
# 15. TEST PREDICTIONS
# ============================================================

test_predictions_raw = iso_forest.predict(
    X_test
)

y_pred = (
    test_predictions_raw == -1
).astype(int)


print("\nTest prediction completed.")


# ============================================================
# 16. TEST PREDICTION DISTRIBUTION
# ============================================================

print("\nPredicted test distribution:")

print(
    pd.Series(y_pred)
    .value_counts()
    .sort_index()
    .rename(
        index={
            0: "Normal",
            1: "Anomaly"
        }
    )
)


# ============================================================
# 17. TEST ANOMALY SCORES
# ============================================================
#
# score_samples():
#
# Higher = more normal
#
# Therefore:
#
# anomaly_score = -score_samples()
#
# Higher = more anomalous
# ============================================================

anomaly_scores = (
    -iso_forest.score_samples(X_test)
)


# ============================================================
# 18. TEST METRICS
# ============================================================

accuracy = accuracy_score(
    y_test,
    y_pred
)

precision = precision_score(
    y_test,
    y_pred,
    zero_division=0
)

recall = recall_score(
    y_test,
    y_pred,
    zero_division=0
)

f1 = f1_score(
    y_test,
    y_pred,
    zero_division=0
)

roc_auc = roc_auc_score(
    y_test,
    anomaly_scores
)


# ============================================================
# 19. TEST RESULTS
# ============================================================

print("\n" + "=" * 70)
print("BASELINE TEST RESULTS")
print("=" * 70)

print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1-score  : {f1:.4f}")
print(f"ROC-AUC   : {roc_auc:.4f}")


# ============================================================
# 20. TEST CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(
    y_test,
    y_pred
)

print("\n" + "=" * 70)
print("TEST CONFUSION MATRIX")
print("=" * 70)

print(cm)


# ============================================================
# 21. TEST CLASSIFICATION REPORT
# ============================================================

print("\nClassification report:")

print(
    classification_report(
        y_test,
        y_pred,
        target_names=[
            "Normal",
            "Anomaly"
        ],
        zero_division=0
    )
)


# ============================================================
# 22. VALIDATION PREDICTIONS
# ============================================================

val_predictions_raw = iso_forest.predict(
    X_val
)

y_val_pred = (
    val_predictions_raw == -1
).astype(int)


# ============================================================
# 23. VALIDATION ANOMALY SCORES
# ============================================================

val_anomaly_scores = (
    -iso_forest.score_samples(X_val)
)


# ============================================================
# 24. VALIDATION METRICS
# ============================================================

val_accuracy = accuracy_score(
    y_val,
    y_val_pred
)

val_precision = precision_score(
    y_val,
    y_val_pred,
    zero_division=0
)

val_recall = recall_score(
    y_val,
    y_val_pred,
    zero_division=0
)

val_f1 = f1_score(
    y_val,
    y_val_pred,
    zero_division=0
)

val_roc_auc = roc_auc_score(
    y_val,
    val_anomaly_scores
)


# ============================================================
# 25. VALIDATION RESULTS
# ============================================================

print("\n" + "=" * 70)
print("VALIDATION RESULTS")
print("=" * 70)

print(f"Accuracy  : {val_accuracy:.4f}")
print(f"Precision : {val_precision:.4f}")
print(f"Recall    : {val_recall:.4f}")
print(f"F1-score  : {val_f1:.4f}")
print(f"ROC-AUC   : {val_roc_auc:.4f}")


# ============================================================
# 26. VALIDATION CONFUSION MATRIX
# ============================================================

val_cm = confusion_matrix(
    y_val,
    y_val_pred
)

print("\nValidation confusion matrix:")
print(val_cm)


# ============================================================
# 27. VALIDATION CLASSIFICATION REPORT
# ============================================================

print("\nValidation classification report:")

print(
    classification_report(
        y_val,
        y_val_pred,
        target_names=[
            "Normal",
            "Anomaly"
        ],
        zero_division=0
    )
)


# ============================================================
# 28. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("ISOLATION FOREST FINAL SUMMARY")
print("=" * 70)

print("\nTEST SET")
print("-" * 30)
print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1-score  : {f1:.4f}")
print(f"ROC-AUC   : {roc_auc:.4f}")

print("\nVALIDATION SET")
print("-" * 30)
print(f"Accuracy  : {val_accuracy:.4f}")
print(f"Precision : {val_precision:.4f}")
print(f"Recall    : {val_recall:.4f}")
print(f"F1-score  : {val_f1:.4f}")
print(f"ROC-AUC   : {val_roc_auc:.4f}")




ISOLATION FOREST ANOMALY DETECTION

Dataset loaded successfully.
Dataset shape: (40010, 36)

Columns:
['e2c_total_bytes', 'e4_entropy_h', 'e5_entropy_c', 'e6_time_char', 'e6b_flow_duration_ms', 'e2_client_size', 'e2_client_record_len', 'e2_server_record_len', 'e3_cert_parsed', 'e1_alg_suite_Unknown(0x11eb)', 'e1_alg_suite_Unknown(0x11ec)', 'e1_alg_suite_mlkem1024', 'e1_alg_suite_mlkem768', 'e1_alg_suite_x25519', 'e1_alg_suite_nan', 'e8_proto_context_TLSv1.2_or_older', 'e8_proto_context_TLSv1.3', 'e9_conn_outcome_Failure', 'e9_conn_outcome_Incomplete', 'e9_conn_outcome_Success', 'e3_sig_alg_Unknown(0x0028)', 'e3_sig_alg_nan', 'e1b_tls_version_SSL3.0', 'e1b_tls_version_TLS1.0', 'e1b_ciphersuite_2', 'e1b_ciphersuite_54', 'e1b_ciphersuite_60', 'e2b_tls_version_SSL3.0', 'e2b_tls_version_TLS1.0', 'e2b_ciphersuite_2', 'e2b_ciphersuite_54', 'e2b_ciphersuite_60', 'label', 'split', 'taxonomy', 'ID']

LABEL DISTRIBUTION
label
0    35000
1     5010
Name: count, dtype: int64

DATASET SPLITS
split
